## Assignment 10

In this assignment, you will get an on-hand experience of utilizing PCA as a dimensionality reduction tool to extract features. 

Specifically,

1. Load the digits data from sklearn.

2. Perform a PCA on the dataset **without** specifying `n_components` and which direction is the main principal component? Namely, along the direction, the variance of sample points is the largest.

3. Compute the cumulative variance ratio of all the components. If we request that the PCA method should preserve at least 50% of the total variance, what is the minimum number of principal components?

4. Choose the best number ($N$) of components by cross-validation. In order to achieve it,first you need to apply the PCA with different $N$ to transform the image data. Then, you are required to apply the logistic regression to do the classification with transformed data as $X$ and the corresponding labels as $Y$. Finally, you can do the cross validation for each $N$.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

In [ ]:
 #1.
digits = load_digits()
data = digits.data
labels = digits.target

In [ ]:
 #2.
pca = PCA().fit(data)
print(pca.explained_variance_)

In [ ]:
max(pca.explained_variance_)

In [ ]:
pca.components_[0]

In [ ]:
 #3.
np.cumsum(pca.explained_variance_ratio_)

In [ ]:
#4
from sklearn.datasets import load_boston
from sklearn.preprocessing import scale
from sklearn.model_selection import cross_val_score 
from sklearn.linear_model import LinearRegression 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression 
sns.set()

In [ ]:
pca = PCA()
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.1, random_state=1)
n = data.shape[1]
for i in range(1, n+1):
    X_reduced_train = pca.fit_transform(scale(X_train))[:, :i] 

from sklearn import linear_model
logistic = linear_model.LogisticRegression(max_iter=5000)
logistic.fit(X_reduced_train, y_train)

In [ ]:
mse = []
# 10-fold cv
score = -cross_val_score(logistic, 
    np.ones((len(X_reduced_train), 1)), y_train,
    cv=10, scoring='neg_mean_squared_error')

mse.append(score.mean())

# Calculate MSE using cv for the 13 components, adding one at a time
for i in range(1, n+1):
    score = -cross_val_score(logistic, X_reduced_train[:, :i], y_train, cv=10, scoring='neg_mean_squared_error')
    mse.append(score.mean())

# Plot results
plt.figure(figsize=(10,6))
plt.plot(mse, marker='o')
plt.xlabel('Number of principal components')
plt.ylabel('MSE')

In [ ]:
a = []
for i in range(1, n+1):
    X_reduced_test = pca.transform(scale(X_test))[:, :i]
    logistic.fit(X_reduced_train[:, :i], y_train)
    pred = logistic. predict(X_reduced_test)
    a.append(mean_squared_error(y_test, pred))

In [ ]:
min(a)

In [ ]:
index = []
for i in range(len(a)):
    if a[i] == min(a): index.append(i)

index